In [ ]:
from pathlib import Path
import json
import requests
import re

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'backend').is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'backend').is_dir():
    raise FileNotFoundError('Could not find the HireAI project root. Run this notebook from within the project directory.')
TEST_PATH = (
    PROJECT_ROOT
    / "ai"
    / "datasets"
    / "processed"
    / "hireai_test.jsonl"
)

print("Test dataset:", TEST_PATH)
print("Exists:", TEST_PATH.exists())

In [ ]:
def load_jsonl(path):
    records = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))

    return records


test_dataset = load_jsonl(TEST_PATH)

print("Test examples:", len(test_dataset))

In [ ]:
OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL_NAME = "qwen3:1.7b"

print("Model:", MODEL_NAME)
print("Ollama:", OLLAMA_URL)

In [ ]:
response = requests.get(
    "http://localhost:11434/api/tags"
)

print("Status code:", response.status_code)

print(
    json.dumps(
        response.json(),
        indent=2
    )
)

In [ ]:
def build_prompt(resume_text):
    return f"""
You are HireAI, an AI system for extracting structured information from resumes.

Extract information from the resume below.

Return ONLY valid JSON.
Do not include explanations.
Do not use Markdown.
Do not use ```.

The JSON must contain exactly these fields:

{{
  "candidate": {{
    "name": null,
    "email": null,
    "phone": null
  }},
  "summary": null,
  "skills": [],
  "years_of_experience": null,
  "job_titles": [],
  "experience": [],
  "education": [],
  "projects": [],
  "certifications": []
}}

Resume:

{resume_text}
""".strip()

In [ ]:
def ask_ollama(resume_text):
    prompt = build_prompt(resume_text)

    payload = {
        "model": MODEL_NAME,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0
        }
    }

    response = requests.post(
        OLLAMA_URL,
        json=payload,
        timeout=180
    )

    response.raise_for_status()

    result = response.json()

    return result["response"]

In [ ]:
sample = test_dataset[0]

print("RESUME")
print("=" * 60)

print(sample["input"])

print("\nMODEL OUTPUT")
print("=" * 60)

model_output = ask_ollama(
    sample["input"]
)

print(model_output)

In [ ]:
def extract_json(text):
    text = text.strip()

    # Remove thinking section if present
    text = re.sub(
        r"<think>.*?</think>",
        "",
        text,
        flags=re.DOTALL
    ).strip()

    # Remove markdown code fences
    text = re.sub(
        r"```json\s*",
        "",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"```\s*",
        "",
        text
    )

    # Find JSON object
    start = text.find("{")
    end = text.rfind("}")

    if start == -1 or end == -1:
        raise ValueError(
            "No JSON object found in model output."
        )

    json_text = text[start:end + 1]

    return json.loads(json_text)

In [ ]:
try:
    parsed_output = extract_json(model_output)

    print(
        json.dumps(
            parsed_output,
            indent=2
        )
    )

except Exception as e:
    print("JSON parsing failed:")
    print(e)

In [ ]:
EXPECTED_FIELDS = [
    "candidate",
    "summary",
    "skills",
    "years_of_experience",
    "job_titles",
    "experience",
    "education",
    "projects",
    "certifications"
]


def validate_output_schema(data):
    if not isinstance(data, dict):
        return False

    for field in EXPECTED_FIELDS:
        if field not in data:
            return False

    if not isinstance(data["candidate"], dict):
        return False

    candidate_fields = [
        "name",
        "email",
        "phone"
    ]

    for field in candidate_fields:
        if field not in data["candidate"]:
            return False

    if not isinstance(data["skills"], list):
        return False

    if not isinstance(data["job_titles"], list):
        return False

    if not isinstance(data["experience"], list):
        return False

    if not isinstance(data["education"], list):
        return False

    if not isinstance(data["projects"], list):
        return False

    if not isinstance(data["certifications"], list):
        return False

    return True

In [ ]:
print(
    "Valid schema:",
    validate_output_schema(parsed_output)
)

In [ ]:
def normalize_text(value):
    if value is None:
        return ""

    return re.sub(
        r"\s+",
        " ",
        str(value).strip().lower()
    )


def normalize_list(values):
    if not isinstance(values, list):
        return set()

    return {
        normalize_text(value)
        for value in values
        if value
    }

In [ ]:
def calculate_field_scores(expected, predicted):

    scores = {}

    expected_candidate = expected["candidate"]
    predicted_candidate = predicted.get(
        "candidate",
        {}
    )

    candidate_fields = [
        "name",
        "email",
        "phone"
    ]

    for field in candidate_fields:

        expected_value = normalize_text(
            expected_candidate.get(field)
        )

        predicted_value = normalize_text(
            predicted_candidate.get(field)
        )

        scores[f"candidate.{field}"] = (
            1 if expected_value == predicted_value else 0
        )

    # Skills
    expected_skills = normalize_list(
        expected["skills"]
    )

    predicted_skills = normalize_list(
        predicted.get("skills", [])
    )

    if expected_skills:
        skill_recall = (
            len(expected_skills & predicted_skills)
            / len(expected_skills)
        )
    else:
        skill_recall = 1.0

    scores["skills"] = skill_recall

    # Years of experience
    scores["years_of_experience"] = (
        1
        if expected["years_of_experience"]
        == predicted.get("years_of_experience")
        else 0
    )

    # Job titles
    expected_titles = normalize_list(
        expected["job_titles"]
    )

    predicted_titles = normalize_list(
        predicted.get("job_titles", [])
    )

    if expected_titles:
        title_score = (
            len(expected_titles & predicted_titles)
            / len(expected_titles)
        )
    else:
        title_score = 1.0

    scores["job_titles"] = title_score

    return scores

In [ ]:
expected = sample["output"]

scores = calculate_field_scores(
    expected,
    parsed_output
)

print(
    json.dumps(
        scores,
        indent=2
    )
)

In [ ]:
baseline_results = []

for index, example in enumerate(test_dataset):

    print(
        f"Processing {index + 1}/{len(test_dataset)}..."
    )

    try:
        raw_output = ask_ollama(
            example["input"]
        )

        predicted = extract_json(
            raw_output
        )

        schema_valid = validate_output_schema(
            predicted
        )

        if schema_valid:
            scores = calculate_field_scores(
                example["output"],
                predicted
            )
        else:
            scores = {}

        baseline_results.append({
            "index": index,
            "schema_valid": schema_valid,
            "scores": scores,
            "raw_output": raw_output,
            "predicted": predicted
        })

    except Exception as e:

        baseline_results.append({
            "index": index,
            "schema_valid": False,
            "scores": {},
            "error": str(e)
        })

print("\nEvaluation completed.")

In [ ]:
successful = [
    result
    for result in baseline_results
    if result["schema_valid"]
]

schema_accuracy = (
    len(successful)
    / len(baseline_results)
)

print(
    f"Schema-valid outputs: "
    f"{len(successful)}/{len(baseline_results)}"
)

print(
    f"Schema accuracy: "
    f"{schema_accuracy * 100:.2f}%"
)

In [ ]:
all_scores = {}

for result in successful:

    for field, score in result["scores"].items():

        if field not in all_scores:
            all_scores[field] = []

        all_scores[field].append(score)


print("BASE QWEN3 RESULTS")
print("=" * 60)

for field, values in all_scores.items():

    average = sum(values) / len(values)

    print(
        f"{field:30s}: "
        f"{average * 100:.2f}%"
    )

In [ ]:
BASELINE_PATH = (
    PROJECT_ROOT
    / "ai"
    / "datasets"
    / "processed"
    / "qwen3_baseline_results.json"
)

with open(
    BASELINE_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        baseline_results,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Saved:")
print(BASELINE_PATH)